# 01 — LLM, Messages, Tool Calling & Agent Loop Fundamentals

## Learning requirements
Trước LangChain abstraction, bạn phải hiểu:
- LLM/chat model chỉ sinh output dựa trên context;
- tool calling là model **đề nghị** gọi function, application mới là nơi execute;
- structured output khác free-text;
- agent là một control loop, không phải một loại model.

## Mental model

```text
User -> Model
          |
          +-- final text ---------> User
          |
          +-- tool call request --> Application executes tool
                                     |
                                     +--> Tool result -> Model -> ...
```

Công thức quan trọng: **LLM != Agent**.

## Vocabulary to master

- token / context window
- system/user/assistant/tool message
- inference
- tool/function schema
- tool arguments
- tool result
- structured output
- streaming
- agent loop
- deterministic workflow vs agentic decision

In [ ]:
# Raw SDK example: deliberately NOT LangChain.
# Gemini is used because it is the course default.
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model=os.getenv("GEMINI_MODEL", "gemini-3.7-flash"),
    contents="Explain tool calling in one short paragraph."
)
print(response.text)

## Tool-calling lifecycle — phải hiểu trước khi code agent

Giả sử model nhận tool:

```python
get_weather(city: str) -> dict
```

Model có thể trả logic tương đương:

```json
{
  "tool": "get_weather",
  "arguments": {"city": "Da Nang"}
}
```

Đây **không phải** kết quả thời tiết. Application phải:

1. validate arguments;
2. authorize action;
3. execute `get_weather`;
4. append result như tool message;
5. gọi model lần nữa.

Đây chính là nền tảng của agent loop.

In [ ]:
# Không cần LLM để hiểu control loop.
# Fake model giúp ta thấy orchestration rõ ràng.
def fake_model(messages):
    last = messages[-1]
    if last["role"] == "user":
        return {"type": "tool_call", "name": "multiply", "args": {"a": 6, "b": 7}}
    if last["role"] == "tool":
        return {"type": "final", "content": f"The answer is {last['content']}"}

def multiply(a: int, b: int) -> int:
    return a * b

messages = [{"role": "user", "content": "What is 6 * 7?"}]

while True:
    result = fake_model(messages)
    if result["type"] == "final":
        print(result["content"])
        break

    tool_result = multiply(**result["args"])
    messages.append({
        "role": "tool",
        "name": result["name"],
        "content": str(tool_result)
    })

## Exercise

Tự sửa loop ở trên để có 2 tools:
- `multiply(a, b)`
- `lookup_exchange_rate(base, quote)`

Bổ sung:
- input validation;
- unknown-tool error;
- maximum loop steps để tránh infinite loop.

## Required output
Một note ngắn trả lời chính xác:
1. Model có execute Python function không?
2. Tool schema dùng để làm gì?
3. Tại sao cần max-steps?
4. Khi nào deterministic workflow tốt hơn agent?

## Done criteria
Chỉ sang LangChain Core khi bạn có thể tự vẽ agent loop từ memory.